In [1]:
import os
import pandas as pd
import numpy as np

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam


In [2]:
# Store Info

from pathlib import Path

seed = 0
np.random.seed(seed)
tf.random.set_seed(seed)




def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


store_cols = [
    "행정동코드",
    "행정동명",
    "상권업종대분류명",
    "상권업종중분류명",
    "위도",
    "경도"
]

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw_data"

store_path = DATA_DIR / "store_info" / "상가(상권)정보_서울.csv"

# store_df = read_csv_kor(store_path, usecols=store_cols)
store_df = read_csv_kor(store_path)  # 전체 컬럼 읽기

# display(store_df.head())
store_df.head()


,상가업소번호,상호명,지점명,상권업종대분류코드,상권업종대분류명,상권업종중분류코드,상권업종중분류명,상권업종소분류코드,상권업종소분류명,표준산업분류코드,...,건물관리번호,건물명,도로명주소,구우편번호,신우편번호,동정보,층정보,호정보,경도,위도
0,MA010120220804265295,60계치킨암사,선사점,I2,음식,I210,기타 간이,I21006,치킨,I56193,...,1174010700105020004015779,암사동정웅빌딩,서울특별시 강동구 상암로3길 8,134877,5241,NaN,1,NaN,127.126859,37.550810
1,MA010120220809658173,성심인력공사,NaN,N1,시설관리·임대,N104,고용 알선,N10401,고용 알선업,N75110,...,1117010700100430059022991,NaN,서울특별시 용산구 한강대로 385,140821,4320,NaN,NaN,NaN,126.972240,37.552803
2,MA010120220808205276,칸토빈,NaN,I2,음식,I212,비알코올,I21201,카페,I56221,...,1150010800200200002002536,공항시장역,서울특별시 강서구 방화동로 30,157240,7619,NaN,1,NaN,126.810493,37.563548
3,MA010120220804370987,까치노래연습장,NaN,R1,예술·스포츠,R104,유원지·오락,R10407,노래방,R91223,...,1135010500101690179002482,NaN,서울특별시 노원구 한글비석로23길 2,139815,1682,NaN,NaN,NaN,127.071218,37.660715
4,MA010120220810170381,금빛주얼리,NaN,G2,소매,G217,시계·귀금속 소매,G21701,시계/귀금속 소매업,G47830,...,1111015100100880001000001,종로주얼리타운,서울특별시 종로구 돈화문로10길 2,110370,3138,NaN,1,NaN,126.991861,37.572331


In [3]:
# Sales Info


def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


sales_cols = [
    "기준_년분기_코드",
    "행정동_코드",
    "행정동_코드_명",
    "서비스_업종_코드",
    "서비스_업종_코드_명",
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
]

sales_path = DATA_DIR / "sales_info" / "서울시(추정매출-행정동).csv"

# sales_df = read_csv_kor(sales_path, usecols=sales_cols)
sales_df = read_csv_kor(sales_path) # 전체 컬럼 읽기

# display(store_df.head())
sales_df.head()


,기준_년분기_코드,행정동_코드,행정동_코드_명,서비스_업종_코드,서비스_업종_코드_명,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,월요일_매출_금액,...,시간대_건수~21_매출_건수,시간대_건수~24_매출_건수,남성_매출_건수,여성_매출_건수,연령대_10_매출_건수,연령대_20_매출_건수,연령대_30_매출_건수,연령대_40_매출_건수,연령대_50_매출_건수,연령대_60_이상_매출_건수
0,20253,11740700,둔촌2동,CS300043,전자상거래업,10751618,35,7526133,3225485,0,...,20,10,20,15,0,0,10,0,15,10
1,20253,11740700,둔촌2동,CS300036,조명용품,8249940,595,5622693,2627247,1485258,...,78,0,430,156,0,10,31,93,144,308
2,20253,11740700,둔촌2동,CS300035,인테리어,661900993,11693,527377785,134523208,131389877,...,973,0,7370,2933,0,417,417,2502,2364,4601
3,20253,11740700,둔촌2동,CS300033,철물점,115789484,1381,111203559,4585925,24534077,...,304,0,933,282,0,49,86,203,288,587
4,20253,11740700,둔촌2동,CS300031,가구,13984669,37,5387612,8597057,462456,...,5,0,9,28,0,0,0,0,5,32


In [4]:
# Population Info


def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


population_cols = [
    "행정동_코드",
    "총_유동인구_수",
    "남성_유동인구_수",
    "여성_유동인구_수",
    "연령대_20_유동인구_수",
    "연령대_30_유동인구_수",
    "연령대_40_유동인구_수",
    "시간대_06_11_유동인구_수",
    "시간대_11_14_유동인구_수",
    "시간대_14_17_유동인구_수",
    "시간대_17_21_유동인구_수",
    "시간대_21_24_유동인구_수",
    "월요일_유동인구_수",
    "화요일_유동인구_수",
    "수요일_유동인구_수",
    "목요일_유동인구_수",
    "금요일_유동인구_수",
    "토요일_유동인구_수",
    "일요일_유동인구_수"
]

population_path = DATA_DIR / "population_info" / "서울시(유동인구-행정동).csv"

if not population_path.exists():
    raise FileNotFoundError(f"파일이 없습니다: {population_path}")

# population_df = read_csv_kor(population_path, usecols=population_cols)
population_df = read_csv_kor(population_path)  # 전체 컬럼 읽기

population_df.head()


,기준_년분기_코드,행정동_코드,행정동_코드_명,총_유동인구_수,남성_유동인구_수,여성_유동인구_수,연령대_10_유동인구_수,연령대_20_유동인구_수,연령대_30_유동인구_수,연령대_40_유동인구_수,...,시간대_14_17_유동인구_수,시간대_17_21_유동인구_수,시간대_21_24_유동인구_수,월요일_유동인구_수,화요일_유동인구_수,수요일_유동인구_수,목요일_유동인구_수,금요일_유동인구_수,토요일_유동인구_수,일요일_유동인구_수
0,20253,11740700,둔촌2동,6677641,3098637,3579004,1281325,759362,997949,1050346,...,734474,1030669,881319,961900,954018,954472,951764,941417,938569,975503
1,20253,11740690,둔촌1동,30002,13846,16156,6659,2516,4352,5649,...,3256,4665,3948,4200,4206,4238,4229,4237,4360,4532
2,20253,11740685,길동,18306303,8294964,10011339,2507243,2213064,2922263,2976562,...,2110430,3039907,2457186,2582521,2589839,2594920,2582295,2599532,2660693,2696503
3,20253,11740660,성내3동,6680704,3100063,3580641,941998,886276,1087042,1167095,...,747308,1070199,885242,944868,945197,948454,938577,946974,963368,993264
4,20253,11740650,성내2동,8182706,3812368,4370337,1106351,1132456,1469108,1282904,...,917727,1347781,1090384,1148652,1138541,1148856,1140149,1148255,1215969,1242282


In [ ]:
# 데이터 컬럼 모아보기

files = {
    "store": store_df,
    "sales": sales_df,
    "population": population_df
}

for name, df in files.items():
    print(f"\n📌 {name} 데이터 컬럼 ")
    print(df.columns)



📌 store 데이터 컬럼 
Index(['상가업소번호', '상호명', '지점명', '상권업종대분류코드', '상권업종대분류명', '상권업종중분류코드',
       '상권업종중분류명', '상권업종소분류코드', '상권업종소분류명', '표준산업분류코드', '표준산업분류명', '시도코드',
       '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드',
       '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지',
       '건물부번지', '건물관리번호', '건물명', '도로명주소', '구우편번호', '신우편번호', '동정보', '층정보',
       '호정보', '경도', '위도'],
      dtype='str')

📌 sales 데이터 컬럼 
Index(['기준_년분기_코드', '행정동_코드', '행정동_코드_명', '서비스_업종_코드', '서비스_업종_코드_명',
       '당월_매출_금액', '당월_매출_건수', '주중_매출_금액', '주말_매출_금액', '월요일_매출_금액',
       '화요일_매출_금액', '수요일_매출_금액', '목요일_매출_금액', '금요일_매출_금액', '토요일_매출_금액',
       '일요일_매출_금액', '시간대_00~06_매출_금액', '시간대_06~11_매출_금액', '시간대_11~14_매출_금액',
       '시간대_14~17_매출_금액', '시간대_17~21_매출_금액', '시간대_21~24_매출_금액', '남성_매출_금액',
       '여성_매출_금액', '연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액',
       '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액', '주중_매출_건수',
       '주말_매출_건수', '월요일_매출_건수', '화요일_매출_건수', '수요일_매출